# 数据指标 Dashboard - 纯数据展示

**功能**: 展示所有数据指标和历史对比，无决策推荐

In [ ]:
import pandas as pd
import numpy as np
import sqlite3
from pathlib import Path
from datetime import datetime
from scipy import stats
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 数据库连接
db_path = Path('../../data/invest.db')
conn = sqlite3.connect(db_path)

# 股票池
stocks = {
    '002648.SZ': '卫星化学',
    '002594.SZ': '比亚迪',
    '000833.SZ': '粤桂股份',
    '300442.SZ': '润泽科技',
    '516120.SH': '化工 50ETF'
}

print("✅ 数据加载完成")

In [ ]:
# 选择股票
selected_code = '002594.SZ'  # 比亚迪
selected_name = stocks[selected_code]
print(f"分析股票：{selected_name} ({selected_code})")

# 读取数据
daily = pd.read_sql_query(
    f"SELECT * FROM stock_daily WHERE ts_code='{selected_code}' ORDER BY trade_date",
    conn
)
daily['trade_date'] = pd.to_datetime(daily['trade_date'], format='%Y%m%d')

print(f"数据量：{len(daily)} 条")
print(f"时间范围：{daily['trade_date'].min().strftime('%Y-%m-%d')} 至 {daily['trade_date'].max().strftime('%Y-%m-%d')}")

In [ ]:
# 图 1: 价格走势 + 历史分位
fig1 = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.05, row_heights=[0.7, 0.3])

# 价格
fig1.add_trace(go.Scatter(
    x=daily['trade_date'],
    y=daily['close'],
    name='收盘价',
    line=dict(color='blue', width=2)
), row=1, col=1)

# 均线
for ma, color, name in [(20, 'orange', 'MA20'), (60, 'green', 'MA60')]:
    ma_line = daily['close'].rolling(ma).mean()
    fig1.add_trace(go.Scatter(
        x=daily['trade_date'],
        y=ma_line,
        name=name,
        line=dict(color=color, width=1, dash='dash')
    ), row=1, col=1)

# 历史分位（滚动 250 日）
rolling_pct = daily['close'].rolling(250).apply(
    lambda x: stats.percentileofscore(x, x.iloc[-1]) / 100 if len(x) > 10 else np.nan
)

fig1.add_trace(go.Scatter(
    x=daily['trade_date'],
    y=rolling_pct,
    name='历史分位',
    line=dict(color='purple', width=2)
), row=2, col=1)

# 分位区间
fig1.add_hrect(y0=0.8, y1=1.0, line_width=0, fillcolor="red", opacity=0.2, row=2, col=1, annotation_text="高分位区")
fig1.add_hrect(y0=0, y1=0.2, line_width=0, fillcolor="green", opacity=0.2, row=2, col=1, annotation_text="低分位区")

fig1.update_layout(
    title=f'{selected_name} - 价格走势 + 历史分位',
    height=700,
    hovermode='x unified'
)

fig1.show()

In [ ]:
# 图 2: 成交量 + 量比
fig2 = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.05, row_heights=[0.7, 0.3])

# 成交量
fig2.add_trace(go.Bar(
    x=daily['trade_date'],
    y=daily['vol'] / 10000,  # 万手
    name='成交量',
    marker_color=np.where(daily['close'] > daily['close'].shift(1), 'red', 'green')
), row=1, col=1)

# 均量线
vol_ma20 = daily['vol'].rolling(20).mean()
fig2.add_trace(go.Scatter(
    x=daily['trade_date'],
    y=vol_ma20 / 10000,
    name='20 日均量',
    line=dict(color='orange', width=2)
), row=1, col=1)

# 量比
volume_ratio = daily['vol'] / vol_ma20
fig2.add_trace(go.Scatter(
    x=daily['trade_date'],
    y=volume_ratio,
    name='量比',
    line=dict(color='blue', width=2)
), row=2, col=1)

fig2.add_hline(y=2, line_dash="dash", line_color="red", row=2, col=1, annotation_text="放量")
fig2.add_hline(y=0.5, line_dash="dash", line_color="green", row=2, col=1, annotation_text="缩量")

fig2.update_layout(
    title=f'{selected_name} - 成交量 + 量比',
    height=600,
    hovermode='x unified'
)

fig2.show()

In [ ]:
# 图 3: 多周期历史分位对比
current_price = daily['close'].iloc[-1]
current_date = daily['trade_date'].iloc[-1]

periods = [
    (5, '5 日'),
    (20, '20 日'),
    (60, '60 日'),
    (250, '250 日')
]

pct_data = []
for days, name in periods:
    hist = daily['close'].tail(days)
    pct = stats.percentileofscore(hist, current_price) / 100
    pct_data.append({
        '周期': name,
        '分位': pct * 100,
        '最小值': hist.min(),
        '最大值': hist.max(),
        '均值': hist.mean(),
        '当前价': current_price
    })

pct_df = pd.DataFrame(pct_data)

fig3 = go.Figure()

fig3.add_trace(go.Bar(
    x=pct_df['周期'],
    y=pct_df['分位'],
    marker_color=['red' if p > 80 else 'green' if p < 20 else 'gray' for p in pct_df['分位']],
    text=[f'{p:.1f}%' for p in pct_df['分位']],
    textposition='outside'
))

fig3.update_layout(
    title=f'{selected_name} - 多周期历史分位对比（截至 {current_date.strftime("%Y-%m-%d")}）',
    xaxis_title='周期',
    yaxis_title='历史分位 (%)',
    yaxis=dict(range=[0, 100]),
    height=500
)

fig3.add_hline(y=80, line_dash="dash", line_color="red", annotation_text="高分位")
fig3.add_hline(y=20, line_dash="dash", line_color="green", annotation_text="低分位")

fig3.show()

# 显示详细数据表
print("\n多周期历史分位详细数据：")
print(pct_df.to_string(index=False))

In [ ]:
# 图 4: 所有股票对比
comparison_data = []

for code, name in stocks.items():
    stock_daily = pd.read_sql_query(
        f"SELECT close, trade_date FROM stock_daily WHERE ts_code='{code}' ORDER BY trade_date",
        conn
    )
    
    if len(stock_daily) == 0:
        continue
    
    stock_daily['trade_date'] = pd.to_datetime(stock_daily['trade_date'], format='%Y%m%d')
    current = stock_daily['close'].iloc[-1]
    
    # 计算各周期分位
    pct_20 = stats.percentileofscore(stock_daily['close'].tail(20), current) / 100
    pct_60 = stats.percentileofscore(stock_daily['close'].tail(60), current) / 100
    pct_250 = stats.percentileofscore(stock_daily['close'].tail(250), current) / 100
    
    comparison_data.append({
        '股票': name,
        '代码': code,
        '当前价': current,
        '20 日分位': pct_20 * 100,
        '60 日分位': pct_60 * 100,
        '250 日分位': pct_250 * 100
    })

comp_df = pd.DataFrame(comparison_data)

fig4 = make_subplots(rows=1, cols=3, subplot_titles=['20 日分位', '60 日分位', '250 日分位'])

for i, col in enumerate(['20 日分位', '60 日分位', '250 日分位'], 1):
    fig4.add_trace(go.Bar(
        x=comp_df['股票'],
        y=comp_df[col],
        marker_color=['red' if p > 80 else 'green' if p < 20 else 'gray' for p in comp_df[col]],
        text=[f'{p:.1f}%' for p in comp_df[col]],
        textposition='outside',
        name=col,
        showlegend=False
    ), row=1, col=i)

fig4.update_layout(
    title='所有股票历史分位对比',
    height=500,
    yaxis=dict(range=[0, 100])
)

fig4.show()

print("\n所有股票对比数据：")
print(comp_df.to_string(index=False))

conn.close()